In [60]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
!ls /content/drive/MyDrive/SentinelAI

'DATASET1 '   device.csv   http.csv   logon.csv


In [42]:
import pandas as pd

base_path = "/content/drive/MyDrive/SentinelAI"

logon = pd.read_csv(f"{base_path}/logon.csv")
device = pd.read_csv(f"{base_path}/device.csv")

logon.head(), device.head()

(                         id                 date          user       pc  \
 0  {Y6O4-A7KC67IN-0899AOZK}  01/04/2010 00:10:37  DTAA/KEE0997  PC-1914   
 1  {O5Y6-O7CJ02JC-6704RWBS}  01/04/2010 00:52:16  DTAA/KEE0997  PC-1914   
 2  {D2D1-C6EB14QJ-2100RSZO}  01/04/2010 01:17:20  DTAA/KEE0997  PC-3363   
 3  {H9W1-X0MC70BT-6065RPAT}  01/04/2010 01:28:34  DTAA/KEE0997  PC-3363   
 4  {H3H4-S5AZ00AZ-9560IYHC}  01/04/2010 01:57:30  DTAA/BJM0992  PC-3058   
 
   activity  
 0    Logon  
 1   Logoff  
 2    Logon  
 3   Logoff  
 4    Logon  ,
                          id                 date          user       pc  \
 0  {S7A7-Y8QZ65MW-8738SAZP}  01/04/2010 07:12:31  DTAA/RES0962  PC-3736   
 1  {G7A8-G1OB94NR-3006NTXH}  01/04/2010 07:35:40  DTAA/BJC0569  PC-2588   
 2  {R3L8-N0LW95FR-8358LLXS}  01/04/2010 08:00:38  DTAA/EMZ0196  PC-1479   
 3  {I2F1-B5FB51FL-3128HBUL}  01/04/2010 08:02:14  DTAA/ZKH0388  PC-1021   
 4  {P7R6-C5TV18CT-1677DWWM}  01/04/2010 08:20:17  DTAA/RES0962  PC-3736   
 

In [43]:
logon.columns

Index(['id', 'date', 'user', 'pc', 'activity'], dtype='object')

In [44]:
logon['date'] = pd.to_datetime(logon['date'])
logon.head()

,id,date,user,pc,activity
0,{Y6O4-A7KC67IN-0899AOZK},2010-01-04 00:10:37,DTAA/KEE0997,PC-1914,Logon
1,{O5Y6-O7CJ02JC-6704RWBS},2010-01-04 00:52:16,DTAA/KEE0997,PC-1914,Logoff
2,{D2D1-C6EB14QJ-2100RSZO},2010-01-04 01:17:20,DTAA/KEE0997,PC-3363,Logon
3,{H9W1-X0MC70BT-6065RPAT},2010-01-04 01:28:34,DTAA/KEE0997,PC-3363,Logoff
4,{H3H4-S5AZ00AZ-9560IYHC},2010-01-04 01:57:30,DTAA/BJM0992,PC-3058,Logon


In [46]:
logon['hour'] = logon['date'].dt.hour

In [47]:
logon['after_hours'] = ((logon['hour'] < 8) | (logon['hour'] > 18)).astype(int)

In [48]:
user_logon = logon.groupby('user').agg(
    total_events=('user', 'count'),
    after_hours_events=('after_hours', 'sum'),
    distinct_pcs=('pc', 'nunique')
).reset_index()

user_logon.head()

,user,total_events,after_hours_events,distinct_pcs
0,DTAA/AAA0371,843,290,16
1,DTAA/AAC0344,690,166,1
2,DTAA/AAC0599,690,178,1
3,DTAA/AAH0734,690,0,1
4,DTAA/AAK0658,690,0,1


In [49]:
device.columns

Index(['id', 'date', 'user', 'pc', 'activity'], dtype='object')

In [50]:
device['date'] = pd.to_datetime(device['date'])
device.head()

,id,date,user,pc,activity
0,{S7A7-Y8QZ65MW-8738SAZP},2010-01-04 07:12:31,DTAA/RES0962,PC-3736,Connect
1,{G7A8-G1OB94NR-3006NTXH},2010-01-04 07:35:40,DTAA/BJC0569,PC-2588,Connect
2,{R3L8-N0LW95FR-8358LLXS},2010-01-04 08:00:38,DTAA/EMZ0196,PC-1479,Connect
3,{I2F1-B5FB51FL-3128HBUL},2010-01-04 08:02:14,DTAA/ZKH0388,PC-1021,Connect
4,{P7R6-C5TV18CT-1677DWWM},2010-01-04 08:20:17,DTAA/RES0962,PC-3736,Disconnect


In [51]:
user_device = device.groupby('user').agg(
    device_events=('user', 'count'),
    distinct_device_pcs=('pc', 'nunique')
).reset_index()

user_device.head()

,user,device_events,distinct_device_pcs
0,DTAA/ABB0272,655,1
1,DTAA/ABS0726,62,1
2,DTAA/ACH0803,516,1
3,DTAA/AFF0760,6,1
4,DTAA/AFH0331,520,1


In [52]:
data = user_logon.merge(user_device, on='user', how='left')

# users with no device activity → fill with 0
data[['device_events', 'distinct_device_pcs']] = (
    data[['device_events', 'distinct_device_pcs']].fillna(0)
)

data.head()

,user,total_events,after_hours_events,distinct_pcs,device_events,distinct_device_pcs
0,DTAA/AAA0371,843,290,16,0.0,0.0
1,DTAA/AAC0344,690,166,1,0.0,0.0
2,DTAA/AAC0599,690,178,1,0.0,0.0
3,DTAA/AAH0734,690,0,1,0.0,0.0
4,DTAA/AAK0658,690,0,1,0.0,0.0


In [53]:
X = data.drop(columns=['user'])

In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [55]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
data['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

data.head()

,user,total_events,after_hours_events,distinct_pcs,device_events,distinct_device_pcs,kmeans_cluster
0,DTAA/AAA0371,843,290,16,0.0,0.0,1
1,DTAA/AAC0344,690,166,1,0.0,0.0,1
2,DTAA/AAC0599,690,178,1,0.0,0.0,1
3,DTAA/AAH0734,690,0,1,0.0,0.0,1
4,DTAA/AAK0658,690,0,1,0.0,0.0,1


In [56]:
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters=3)
data['agg_cluster'] = agg.fit_predict(X_scaled)

In [58]:
data.groupby('kmeans_cluster').mean(numeric_only=True)

,total_events,after_hours_events,distinct_pcs,device_events,distinct_device_pcs,agg_cluster
kmeans_cluster,,,,,,
0,974.052863,241.180617,28.118943,287.867841,1.000000,0.0
1,748.252300,214.704336,7.291721,0.000000,0.000000,2.0
2,4920.750000,2528.833333,877.916667,26.833333,0.083333,1.0


In [61]:
import torch

# convert numpy array to torch tensor
X_torch = torch.tensor(X_scaled, dtype=torch.float32)

# number of clusters
k = 3

# randomly initialize centroids
indices = torch.randperm(X_torch.size(0))[:k]
centroids = X_torch[indices]

# simple k-means iterations
for _ in range(20):
    # compute distances
    distances = torch.cdist(X_torch, centroids)
    labels = torch.argmin(distances, dim=1)

    # update centroids
    for i in range(k):
        centroids[i] = X_torch[labels == i].mean(dim=0)

# store pytorch cluster labels
data['pytorch_kmeans_cluster'] = labels.numpy()

In [62]:
data['pytorch_kmeans_cluster'].value_counts()

,count
pytorch_kmeans_cluster,
1,1000


In [59]:
data.groupby('agg_cluster').mean(numeric_only=True)

,total_events,after_hours_events,distinct_pcs,device_events,distinct_device_pcs,kmeans_cluster
agg_cluster,,,,,,
0,974.052863,241.180617,28.118943,287.867841,1.000000,0.0
1,4920.750000,2528.833333,877.916667,26.833333,0.083333,2.0
2,748.252300,214.704336,7.291721,0.000000,0.000000,1.0
